# FastAPI 简介

FastAPI 是一个用于构建 Web API 的现代 Python 框架。它基于 Python 类型提示完成参数解析、数据校验和文档生成，具有开发快、性能高、易于维护等特点。

## 核心概念

- **路由**：使用 `@app.get()`、`@app.post()` 等装饰器，把 URL 和 Python 函数关联起来。
- **类型校验**：声明 `item_id: int` 后，FastAPI 会自动检查输入；类型不正确时返回清晰的错误信息。
- **数据模型**：可以使用 Pydantic 模型描述请求体和响应数据。
- **自动文档**：服务启动后，可通过 `/docs` 查看 Swagger UI，通过 `/redoc` 查看 ReDoc。
- **异步支持**：可以使用 `async def` 编写适合 I/O 密集型任务的接口。

## 最小示例

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/items/{item_id}")
def read_item(item_id: int, q: str | None = None):
    return {"item_id": item_id, "q": q}
```

安装并启动：

```bash
pip install fastapi uvicorn
uvicorn main:app --reload
```

请求 `GET /items/42?q=book` 时，FastAPI 会匹配路由、把 `42` 校验并转换成整数，然后返回 JSON：`{"item_id": 42, "q": "book"}`。


In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# 创建 FastAPI 应用
app = FastAPI(title="FastAPI 演示")

# 定义一个带路径参数和查询参数的 GET 接口
@app.get("/items/{item_id}")
def read_item(item_id: int, q: str | None = None):
    return {"item_id": item_id, "q": q}

# 在 Notebook 中模拟客户端请求，无需启动 Web 服务器
client = TestClient(app)
response = client.get("/items/42", params={"q": "book"})

print("状态码：", response.status_code)
print("响应数据：", response.json())


状态码： 200
响应数据： {'item_id': 42, 'q': 'book'}


In [6]:
from typing import Optional

from fastapi import FastAPI
import uvicorn
from pydantic import BaseModel, Field

app = FastAPI()

students = [{"id": 1, "name": "张三", "age": 18, "class": "高三（1）班级"},
            {"id": 2, "name": "王五", "age": 19, "class": "高三（2）班级"}]

@app.get("/students")
def get_students():
    return students

class Student(BaseModel):
    name: Optional[str] = Field(description="姓名", default=None)
    age: int
    class_: str = Field(description="班级", default=None, alias="class")

@app.post("/students")
def add_student(student: Student):
    new_student = student.model_dump()
    new_student["id"] = max(s["id"] for s in students) + 1
    students.append(new_student)
    return new_student

@app.put("/students/{student_id}")
def update_student(student_id: int, student: Student):
    for i in range(len(students)):
        if students[i]["id"] == student_id:
            students[i] = student.model_dump()
            students[i]["id"] = student_id
            return students[i]
    return {"message": "没有此学生"}

@app.delete("/students/{student_id}")
def delete_student(student_id: int):
    for i in range(len(students)):
        if students[i]["id"] == student_id:
            del students[i]
            return {"message": f"ID{student_id}学生删除成功"}
    return {"message": f"没有ID{student_id}此学生"}

from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/students")
print("【GET    /students    】状态码：", response.status_code)
print("响应数据：", response.json())
print("-" * 60)

response = client.post("/students", json={"name": "李四", "age": 18, "class": "高三（3）班级"})
print("【POST   /students    】状态码：", response.status_code)
print("响应数据：", response.json())
print("-" * 60)

response = client.put("/students/1", json={"name": "张三丰", "age": 20, "class": "高三（1）班级"})
print("【PUT    /students/1  】状态码：", response.status_code)
print("响应数据：", response.json())
print("-" * 60)

response = client.delete("/students/2")
print("【DELETE /students/2  】状态码：", response.status_code)
print("响应数据：", response.json())
print("-" * 60)

response = client.post("/students", json={"name": "测试", "age": "abc", "class": "高三（1）班级"})
print("【校验 age 传 'abc'  】状态码：", response.status_code)
print("响应数据：", response.json())

【GET    /students    】状态码： 200
响应数据： [{'id': 1, 'name': '张三', 'age': 18, 'class': '高三（1）班级'}, {'id': 2, 'name': '王五', 'age': 19, 'class': '高三（2）班级'}]
------------------------------------------------------------
【POST   /students    】状态码： 200
响应数据： {'name': '李四', 'age': 18, 'class_': '高三（3）班级', 'id': 3}
------------------------------------------------------------
【PUT    /students/1  】状态码： 200
响应数据： {'name': '张三丰', 'age': 20, 'class_': '高三（1）班级', 'id': 1}
------------------------------------------------------------
【DELETE /students/2  】状态码： 200
响应数据： {'message': 'ID2学生删除成功'}
------------------------------------------------------------
【校验 age 传 'abc'  】状态码： 422
响应数据： {'detail': [{'type': 'int_parsing', 'loc': ['body', 'age'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


## 单元格 3 代码说明：学生信息管理 API

这个示例把普通 Python 函数注册成可以通过 HTTP 访问的接口，演示 RESTful 风格的学生信息增、删、改、查，以及 Pydantic 参数校验。

### 学习目标

1. 理解 API 如何把 Python 函数变成可访问的接口。
2. 掌握 `GET`、`POST`、`PUT`、`DELETE` 四种常用 HTTP 方法。
3. 使用 Pydantic 描述请求体并自动校验数据。
4. 使用 FastAPI 自动文档或 `TestClient` 测试接口。

### 核心组件

- `FastAPI` 负责路由分发，把请求交给对应的 Python 函数。
- Uvicorn 是 ASGI 服务器，负责监听端口并收发网络请求；FastAPI 本身不监听端口。
- `BaseModel` 用于定义请求体结构，`Field` 用于设置字段说明、默认值和别名。
- `Optional[str]` 表示字符串字段可以不提供。
- `students` 列表用于模拟数据库，程序重新运行后数据会恢复。

安装依赖：

```bash
pip install fastapi==0.124.4 uvicorn==0.38.0
```

### RESTful 接口

| 操作 | HTTP 方法 | 路径 | 参数位置 |
|---|---|---|---|
| 查询全部 | `GET` | `/students` | 无 |
| 新增一个 | `POST` | `/students` | 请求体 JSON |
| 修改一个 | `PUT` | `/students/{student_id}` | 路径参数和请求体 |
| 删除一个 | `DELETE` | `/students/{student_id}` | 路径参数 |

RESTful 的核心是：URL 使用名词表示资源，HTTP 方法表示操作。例如推荐使用 `GET /students/1`，而不是把动作写进 `/getStudentById?id=1`。`PUT` 表示整条数据全量更新，`PATCH` 通常表示只更新部分字段。

### 数据模型与校验

`Student` 模型规定：`name` 可以省略，`age` 是必填整数，`class_` 通过 `alias="class"` 接收 JSON 中的 `class` 字段。FastAPI 会自动读取 JSON、校验字段、进行类型转换，并在 `/docs` 中生成字段说明。若把 `age` 传成 `"abc"`，会自动返回 `422`。

### 各接口的处理过程

- 查询：直接返回 `students`，FastAPI 自动把 Python 列表和字典转换成 JSON。
- 新增：使用 `student.model_dump()` 转成字典，以当前最大 ID 加一生成新 ID，再追加到列表。
- 修改：使用路径参数 `student_id` 找到学生，用请求体数据整条替换，然后补回原 ID。
- 删除：根据 `student_id` 找到列表位置并删除；不存在时返回提示消息。

### 测试方式

代码最后使用 `TestClient` 在 Notebook 内依次模拟查询、新增、修改、删除和错误数据校验，无需启动网络服务器。若把应用保存为 `main.py` 并使用 Uvicorn 启动，可访问 Swagger UI：<http://127.0.0.1:8000/docs>，或 ReDoc：<http://127.0.0.1:8000/redoc>。也可以使用 Postman 或 Apifox 测试。

## 同步与异步

### 一句话区别

- **同步（Synchronous）**：一件事做完才能做下一件，期间"阻塞"等待。
- **异步（Asynchronous）**：等结果的同时可以先去干别的，不阻塞。

### 生活类比：去餐厅吃饭

| 场景 | 同步 | 异步 |
|---|---|---|
| 点餐 | 站在柜台前等厨师做好，**期间什么都干不了** | 点完单拿个号，**先去逛街**，好了叫你 |
| 特点 | 简单、顺序清晰 | 复杂一点，但**不浪费时间** |

### 为什么 Web 服务需要异步？

一个 Web 服务的请求，大部分时间都花在"等"上面：

```text
请求进来 → 读数据库(等) → 调其他服务(等) → 读文件(等) → 返回结果
             ↑ 这几步是 I/O（输入输出），CPU 基本在闲着等
```

如果每个请求都**同步阻塞**，一个人占着位置等，来了 100 个请求就得排队 100 个位置。
**异步**的好处是：等待 I/O 的时候，CPU 可以切换去处理别的请求，同样的资源能扛更多并发。

### FastAPI 里的同步 vs 异步

```python
# 同步接口：普通 def
@app.get("/sync")
def read_sync():
    time.sleep(2)          # 阻塞 2 秒
    return {"msg": "同步"}

# 异步接口：async def
@app.get("/async")
async def read_async():
    await asyncio.sleep(2) # 不阻塞，让出控制权
    return {"msg": "异步"}
```

| | 同步 `def` | 异步 `async def` |
|---|---|---|
| 写法 | 普通函数 | `async def` + `await` |
| 适合 | CPU 密集（计算、模型推理） | I/O 密集（数据库、网络请求、读文件） |
| 处理方式 | FastAPI 丢进**线程池**跑 | 在**事件循环**里跑 |
| 卡住影响 | 占用一个线程，但线程池可复用 | 卡住会阻塞整个事件循环（危险） |

### 什么时候用哪个？

- **I/O 密集**（等网络、等数据库、等文件）→ 用 `async def`，并发能力强。
- **CPU 密集**（YOLO 推理、图像处理、大量计算）→ 用普通 `def`，让 FastAPI 丢线程池跑。
- **不确定** → 用普通 `def` 最安全（FastAPI 自动放线程池，不会阻塞主流程）。

### 常见误区

1. ❌ 在 `async def` 里写 `time.sleep(2)` —— 这是**同步阻塞**，会把整个事件循环卡死；
   应该用 `await asyncio.sleep(2)`。
2. ❌ 以为 `async def` 一定更快 —— 它只是**不浪费等待时间**，单请求本身并不更快。
3. ✅ 项目里 YOLO 推理这种 CPU 密集任务，就用普通 `def` 写接口最合适。


## 容器创建与部署流程总结（YOLOv8n ONNX 推理服务）

本节课把训练好的 **YOLOv8n ONNX 模型** 和推理环境封装进 Docker 容器，对外提供 FastAPI 接口。这就是企业部署的标准链路：

```text
训练模型(best.pth) → 导出 ONNX → 编写推理服务(FastAPI) → Docker 打包 → 容器运行 → HTTP 对外服务
```

### 一、整体架构

```text
浏览器/任意程序
     │  POST /detect 上传图片
     ▼
┌─────────────────────────────┐
│  Docker 容器 (端口 8001)    │
│  FastAPI  (app/main.py)     │
│  onnxruntime (yolov8n.onnx) │
│  OpenCV 预处理 + NMS 后处理  │
└─────────────────────────────┘
     │
     ▼
  返回 JSON 检测结果 / 标注图
```

### 二、项目结构

```text
week15/yolo_onnx_service/
├── app/
│   ├── yolo_detector.py   # ONNX 推理封装：letterbox 预处理 + NMS 后处理
│   ├── main.py            # FastAPI 接口 + 托管前端页面
│   └── static/index.html  # Web 前端（上传图片可视化检测）
├── models/yolov8n.onnx    # 模型（打进镜像 = 封装）
├── requirements.txt       # 依赖清单
├── Dockerfile             # 镜像构建脚本
├── docker-compose.yml     # 容器编排（端口映射等）
├── client_test.py         # 本地测试客户端
└── README.md
```

### 三、Dockerfile 关键点

```dockerfile
FROM python:3.11-slim          # 1. 选轻量基础镜像（部署阶段不需要 CUDA/PyTorch）
WORKDIR /app
COPY requirements.txt .        # 2. 先装依赖 → 利用层缓存加速二次构建
RUN pip install -r requirements.txt
COPY app/ ./app/               # 3. 拷贝代码
COPY models/ ./models/         # 4. 拷贝模型（真正"打包"）
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### 四、核心概念

| 概念 | 说明 | 类比 |
|---|---|---|
| **镜像 (Image)** | 只读的"安装包"，包含代码+环境+模型 | 光盘/安装包 |
| **容器 (Container)** | 镜像跑起来的实例，可启停 | 用光盘装好的电脑 |
| **Dockerfile** | 描述如何一步步构建镜像 | 安装说明书 |
| **docker-compose.yml** | 描述如何启动容器（端口/卷/环境变量） | 一键启动脚本 |
| **端口映射** | `8001:8000` = 宿主机 8001 → 容器内 8000 | 门牌号 |
| **层缓存** | 每行指令是一层，没变的层直接复用 | 增量更新 |

### 五、部署步骤（完整流程）

```powershell
# 1. 准备模型（从 week13 复制）
Copy-Item ..\..\week13\onnx_models\yolov8n.onnx models\

# 2. 构建镜像并启动容器（首次较慢，后续走缓存很快）
cd d:\project\step1\week15\yolo_onnx_service
docker compose up -d --build

# 3. 验证
docker compose ps                          # 状态应为 Up (healthy)
D:/project/step1/env/python.exe client_test.py ..\..\week13\img.jpg

# 4. 使用
浏览器打开 http://127.0.0.1:8001/          # Web 前端界面
浏览器打开 http://127.0.0.1:8001/docs      # Swagger 接口文档

# 5. 日常运维
docker compose logs -f                     # 看日志
docker compose down                        # 停止并删除容器
```

### 六、接口一览

| 方法 | 路径 | 说明 |
|------|------|------|
| GET | `/` | Web 前端界面 |
| GET | `/health` | 健康检查（Docker 用它判断容器是否存活） |
| POST | `/detect` | 上传图片 → 返回检测框 JSON |
| POST | `/annotate` | 上传图片 → 返回画好框的标注图 |

### 七、经验与坑

1. **开发镜像 vs 部署镜像**：开发用 PyTorch 全量镜像（12.3GB，方便训练调试）；部署用 python-slim + onnxruntime（703MB）。模型已是 ONNX，推理不需要 PyTorch。
2. **前端参数传递**：FastAPI 中 `conf: float = 0.25` 是**查询参数**，前端必须拼在 URL `?conf=` 后面；放 FormData 里后端收不到（踩过的坑）。
3. **代码改动生效**：改代码后必须 `docker compose up -d --build` 重新构建，因为代码是 COPY 进镜像的（不是挂载）。
4. **端口冲突**：8000 常被占用，所以对外映射到 8001；本地跑和 Docker 跑别同时占同一端口。
5. **健康检查**：Dockerfile 里配了 `HEALTHCHECK`，容器状态变成 healthy 才算真正就绪。
